<a href="https://colab.research.google.com/github/AnikKushwaha/AnikKushwaha/blob/master/Ticket_7272638.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install --quiet ultralytics sentence-transformers faiss-cpu opencv-python-headless Pillow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 10.8 MB/s eta 0:00:00


In [2]:

import os
import sqlite3
import json
import time
from typing import List, Tuple, Dict, Optional

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception as e:
    YOLO_AVAILABLE = False

from sentence_transformers import SentenceTransformer, util

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
DEVICE: cpu


In [3]:
DB_PATH = "products.db"

def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            sku TEXT UNIQUE,
            brand TEXT,
            category TEXT,
            image_path TEXT,
            embedding_json TEXT
        )
    ''')
    conn.commit()
    conn.close()

def add_product(name, sku, brand, category, image_path, embedding: List[float], db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    emb_json = json.dumps([float(x) for x in embedding])
    cur.execute('''
        INSERT OR REPLACE INTO products (name, sku, brand, category, image_path, embedding_json)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (name, sku, brand, category, image_path, emb_json))
    conn.commit()
    conn.close()

def load_products(db_path=DB_PATH) -> List[Dict]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('SELECT id, name, sku, brand, category, image_path, embedding_json FROM products')
    rows = cur.fetchall()
    conn.close()
    products = []
    for r in rows:
        emb = json.loads(r[6]) if r[6] else None
        products.append({
            "id": r[0], "name": r[1], "sku": r[2], "brand": r[3],
            "category": r[4], "image_path": r[5], "embedding": emb
        })
    return products

init_db()


In [ ]:

detector = None
if YOLO_AVAILABLE:
    try:
        detector = YOLO("yolov8n.pt")  # ultralytics will auto-download if needed
        print("YOLOv8 loaded.")
    except Exception as e:
        print("YOLO load failed:", e)
        detector = None
else:
    print("Ultralytics YOLO not available. Will use OpenCV fallback for detection.")

# Embeddings model (CLIP-style)
embedding_model = SentenceTransformer("clip-ViT-B-32")  # small/fast CLIP model
embedding_model = embedding_model.to(DEVICE)
print("Embedding model ready.")


YOLOv8 loaded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

0_CLIPModel/pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

0_CLIPModel/model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Embedding model ready.


In [ ]:
def read_image(path_or_bytes):
    if isinstance(path_or_bytes, (bytes, bytearray)):
        nparr = np.frombuffer(path_or_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    else:
        img = cv2.imread(path_or_bytes, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Unable to read image {path_or_bytes}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def resize_max(img, max_size=800):
    h, w = img.shape[:2]
    if max(h, w) <= max_size:
        return img
    scale = max_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

def crop_with_box(img, box: Tuple[int, int, int, int], margin=0.02):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    dx = int((x2 - x1) * margin)
    dy = int((y2 - y1) * margin)
    x1 = max(0, x1 - dx)
    y1 = max(0, y1 - dy)
    x2 = min(w, x2 + dx)
    y2 = min(h, y2 + dy)
    return img[y1:y2, x1:x2]

def apply_grabcut(img, rect=None, iter_count=5):
    mask = np.zeros(img.shape[:2], np.uint8)
    bgdModel = np.zeros((1,65),np.float64)
    fgdModel = np.zeros((1,65),np.float64)
    if rect is None:
        h, w = img.shape[:2]
        rect = (int(w*0.05), int(h*0.05), int(w*0.9), int(h*0.9))
    x, y, wR, hR = rect
    try:
        cv_img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        cv2.grabCut(cv_img, mask, rect, bgdModel, fgdModel, iter_count, cv2.GC_INIT_WITH_RECT)
        mask2 = np.where((mask==2)|(mask==0), 0,1).astype('uint8')
        img_fg = img * mask2[:,:,None]
        return img_fg
    except Exception as e:
        return img

In [ ]:
def detect_products(img: np.ndarray, conf_thresh=0.3):
    """
    Returns list of detections: each detection is dict {box:[x1,y1,x2,y2], confidence, class_id}
    """
    img_for_det = resize_max(img, max_size=1024)
    h, w = img_for_det.shape[:2]
    detections = []
    if detector is not None:
        results = detector.predict(source=img_for_det, imgsz=640, conf=conf_thresh, max_det=10)
        res = results[0]
        boxes = res.boxes
        for box in boxes:
            xyxy = box.xyxy.cpu().numpy().astype(int)[0]  # [x1,y1,x2,y2]
            conf = float(box.conf.cpu().numpy())
            cls_id = int(box.cls.cpu().numpy()) if box.cls is not None else None
            scale_x = img.shape[1] / w
            scale_y = img.shape[0] / h
            x1, y1, x2, y2 = xyxy
            x1 = int(x1 * scale_x); x2 = int(x2 * scale_x)
            y1 = int(y1 * scale_y); y2 = int(y2 * scale_y)
            detections.append({"box":[x1,y1,x2,y2], "confidence":conf, "class_id":cls_id})
    else:
        gray = cv2.cvtColor(img_for_det, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (5,5), 0)
        _, th = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x,y,wc,hc = cv2.boundingRect(cnt)
            area = wc*hc
            if area < 1000:
                continue

            scale_x = img.shape[1] / w
            scale_y = img.shape[0] / h
            x1 = int(x * scale_x); y1 = int(y * scale_y)
            x2 = int((x+wc) * scale_x); y2 = int((y+hc) * scale_y)
            detections.append({"box":[x1,y1,x2,y2], "confidence": 0.5, "class_id": None})
    return detections


In [ ]:
def embed_image_pil(pil_img, batch_size=1):
    emb = embedding_model.encode(pil_img, convert_to_numpy=True, show_progress_bar=False, batch_size=batch_size, device=DEVICE, normalize_embeddings=True)

    if emb.ndim == 2 and emb.shape[0] == 1:
        emb = emb[0]
    return emb

def cosine_similarity(a, b):
    a = np.array(a); b = np.array(b)
    if np.linalg.norm(a)==0 or np.linalg.norm(b)==0:
        return  -1.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


In [ ]:
def build_product_catalog_from_folder(folder_path, db_path=DB_PATH):

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    for fname in tqdm(files, desc="Indexing product images"):
        path = os.path.join(folder_path, fname)

        name = os.path.splitext(fname)[0]
        sku = name

        pil = Image.open(path).convert("RGB")
        emb = embed_image_pil(pil)

        add_product(name=name, sku=sku, brand=None, category=None, image_path=path, embedding=emb, db_path=db_path)

In [4]:

def recognize_products_in_image(input_image_path: str,
                                db_path=DB_PATH,
                                detection_conf=0.25,
                                similarity_threshold=0.35,
                                top_k=3,
                                use_grabcut=True):
    """
    Returns list of results for each detected product:
    {
        'box': [x1,y1,x2,y2],
        'crop_path': path (optional),
        'candidates': [
            {'id', 'name','sku','brand','category','similarity'}
        ],
        'status': 'matched'|'low_confidence'|'no_candidates'
    }
    """
    img = read_image(input_image_path)
    detections = detect_products(img, conf_thresh=detection_conf)
    if not detections:

        detections = [{"box":[0,0,img.shape[1], img.shape[0]], "confidence":1.0, "class_id":None}]
    products = load_products(db_path)
    db_embeddings = []
    db_meta = []
    for p in products:
        if p["embedding"] is None:
            continue
        db_embeddings.append(np.array(p["embedding"], dtype=np.float32))
        db_meta.append(p)
    if len(db_embeddings) > 0:
        db_embeddings = np.vstack(db_embeddings)
    else:
        db_embeddings = np.zeros((0,512), dtype=np.float32)  # empty

    results = []
    for det in detections:
        box = det["box"]
        crop = crop_with_box(img, box, margin=0.03)
        if use_grabcut:
            try:
                crop_fg = apply_grabcut(crop)
                if np.sum(crop_fg) < 100:
                    crop_to_use = crop
                else:
                    crop_to_use = crop_fg
            except Exception:
                crop_to_use = crop
        else:
            crop_to_use = crop
        pil_crop = Image.fromarray(crop_to_use.astype('uint8'))
        emb = embed_image_pil(pil_crop)

        candidates = []
        if db_embeddings.shape[0] > 0:
            for idx, db_emb in enumerate(db_embeddings):
                sim = cosine_similarity(emb, db_emb)
                candidates.append((sim, db_meta[idx]))
            candidates.sort(key=lambda x: x[0], reverse=True)
            top = candidates[:top_k]
            formatted = []
            for sim, meta in top:
                formatted.append({
                    "id": meta["id"],
                    "name": meta["name"],
                    "sku": meta["sku"],
                    "brand": meta["brand"],
                    "category": meta["category"],
                    "image_path": meta["image_path"],
                    "similarity": float(sim)
                })
            if formatted and formatted[0]["similarity"] >= similarity_threshold:
                status = "matched"
            elif formatted and formatted[0]["similarity"] > 0:
                status = "low_confidence"
            else:
                status = "no_candidates"
        else:
            formatted = []
            status = "no_candidates"

        results.append({
            "box": box,
            "confidence": float(det.get("confidence", 0.0)),
            "candidates": formatted,
            "status": status
        })
    return results
